# 02 — Lectura de archivos: CSV, JSON, Parquet

En la vida real los datos **nunca** nacen como una lista de Python.
Acá vas a leer los tres formatos más comunes y vas a ver por qué Parquet
le gana a CSV en todos los escenarios.

In [ ]:
from pyspark.sql import SparkSession
import time

spark = (
    SparkSession.builder
    .appName("02_lectura_archivos")
    .master("spark://spark-master:7077")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## 1. Generar datos de prueba

Vamos a generar 1M de filas sintéticas para hacer comparaciones reales.

In [ ]:
import random
from datetime import datetime, timedelta
from pathlib import Path

random.seed(42)
out_dir = Path("/home/jovyan/work/data/sample/ventas")
out_dir.mkdir(parents=True, exist_ok=True)

ciudades = ["Bogotá", "Medellín", "Cali", "Barranquilla", "Cartagena"]
productos = ["A", "B", "C", "D", "E"]
base = datetime(2026, 1, 1)

with (out_dir / "ventas.csv").open("w") as f:
    f.write("id,fecha,ciudad,producto,cantidad,precio\n")
    for i in range(1_000_000):
        fecha = base + timedelta(days=random.randint(0, 365))
        f.write(
            f"{i},{fecha.date()},{random.choice(ciudades)},"
            f"{random.choice(productos)},{random.randint(1, 10)},"
            f"{random.uniform(1000, 50000):.2f}\n"
        )
print("Generado ventas.csv con 1M filas")

## 2. CSV: simple, pero no escalable

In [ ]:
t = time.time()
csv_df = spark.read.csv(
    str(out_dir / "ventas.csv"),
    header=True,
    inferSchema=True,  # ojo: en prod, schema explícito
)
csv_count = csv_df.count()
print(f"CSV: leído y contado en {time.time() - t:.2f}s ({csv_count} filas)")
csv_df.printSchema()

## 3. Parquet: 3-5x más rápido para leer + comprime mejor

In [ ]:
# Escribir a Parquet
t = time.time()
csv_df.write.mode("overwrite").parquet(str(out_dir / "ventas_parquet"))
print(f"Escritura Parquet: {time.time() - t:.2f}s")

# Tamaño en disco
csv_size = (out_dir / "ventas.csv").stat().st_size / 1024 / 1024
parq_size = sum(
    p.stat().st_size for p in (out_dir / "ventas_parquet").rglob("*.parquet")
) / 1024 / 1024
print(f"CSV: {csv_size:.1f} MB")
print(f"Parquet: {parq_size:.1f} MB ({csv_size / parq_size:.1f}x más chico)")

In [ ]:
# Lectura Parquet
t = time.time()
parq_df = spark.read.parquet(str(out_dir / "ventas_parquet"))
parq_count = parq_df.count()
print(f"Parquet: leído y contado en {time.time() - t:.2f}s ({parq_count} filas)")

## 4. La magia columnar: leer SOLO lo que necesitás

In [ ]:
# CSV: no importa qué pidas, lee todo
t = time.time()
csv_df.select("ciudad").distinct().collect()
print(f"CSV: distinct ciudades = {time.time() - t:.2f}s")

# Parquet: solo lee la columna ciudad
t = time.time()
parq_df.select("ciudad").distinct().collect()
print(f"Parquet: distinct ciudades = {time.time() - t:.2f}s")

## 5. JSON: lo más común en APIs

**Ojo con `inferSchema=True` en JSON**: Spark tiene que leer TODO el
archivo para inferir. En prod, schema explícito siempre.

In [ ]:
import json
with (out_dir / "ventas.json").open("w") as f:
    for i in range(1000):
        json.dump({
            "id": i,
            "ciudad": random.choice(ciudades),
            "monto": random.uniform(1000, 50000),
            "tags": ["web", "mobile"][i % 2:][:1],
        }, f)
        f.write("\n")

json_df = spark.read.json(
    str(out_dir / "ventas.json"),
    multiLine=False,  # una fila = un JSON
    schema="id LONG, ciudad STRING, monto DOUBLE, tags ARRAY<STRING>",
)
json_df.printSchema()
json_df.show(5, truncate=False)

## 6. Schemas en producción (DDL string)

In [ ]:
ventas_schema = """
    id INT,
    fecha DATE,
    ciudad STRING,
    producto STRING,
    cantidad INT,
    precio DECIMAL(10, 2)
"""

df = spark.read.csv(
    str(out_dir / "ventas.csv"),
    header=True,
    schema=ventas_schema,
)
df.printSchema()

## ✅ Checkpoint

- [x] Sé leer CSV, JSON, Parquet.
- [x] Entiendo por qué Parquet es mejor que CSV para casi todo.
- [x] Uso schema explícito en producción (no `inferSchema`).

**Siguiente**: `03_transforms_comunes.ipynb` — las 30 operaciones que vas
a usar el 95% del tiempo.